测验 1：手算局部倒数与反向传播（对应第 5 章）

**背景：** 假设我们有一个最微型的神经网络计算节点，包含一次加权求和（类似微型 Affine 层）和一个 ReLU 激活函数。

**计算过程：**
1. 加权求和：$a = w_1 \cdot x_1 + w_2 \cdot x_2 + b$
2. 激活函数：$y = \text{ReLU}(a)$

**已知当前输入数据为：**
* $x_1 = 2,\ x_2 = -1$
* $w_1 = 1,\ w_2 = 3,\ b = -1$
* 假设从下一层传回来的**上游全局梯度（$\frac{\partial L}{\partial y}$）为 `10`**。

**你的任务（手算）：**
1. 正向传播算出的 $a$ 和 $y$ 分别是多少？
2. 利用链式法则，从后向前倒推，计算出两个权重参数的梯度 $\frac{\partial L}{\partial w_1}$ 和 $\frac{\partial L}{\partial w_2}$。

测验 2：动量法（Momentum）模拟（对应第 6 章）

**背景：** 我们来对比一下最基础的 SGD 和带有“惯性”的 Momentum 优化器在起步时的表现。
假设我们的损失函数极其简单：$L(w) = \frac{1}{2} w^2$。
显然，它的梯度（导数）就是 $w$ 本身（即 $\nabla L = w$）。
初始权重 $w_0 = 10$，学习率 $\eta = 0.1$。

**你的任务（手算）：**
1. **使用 SGD**：更新公式为 $w \leftarrow w - \eta \cdot \nabla L$。请算出第一步更新后的 $w_1$ 和第二步更新后的 $w_2$。
2. **使用 Momentum**：动量系数 $\alpha = 0.9$，初始速度 $v_0 = 0$。更新公式为：
   $$v \leftarrow \alpha \cdot v - \eta \cdot \nabla L$$
   $$w \leftarrow w + v$$
   请算出第一步的 $w_1$ 和第二步的 $w_2$。
3. **观察**：比较两种方法第二步走完后的 $w_2$，谁下降得更快？为什么？

3：因为 Momentum 带有“惯性”（动量）。在第二步时，它不仅包含了当前位置的新梯度（$-0.9$），还保留了第一步积累下来的速度（$-0.9$），两者叠加导致第二步迈出的步伐更大（速度从 $-1$ 加速到了 $-1.8$）。

测验3：编程实现 Dropout 层（对应第 6 章）
背景： Dropout 是一种强大的抑制过拟合的方法。它的原理是：在训练时，以设定的概率 ratio 随机让一部分神经元“失活”（输出置为 0）；在测试时，所有神经元都工作，但为了保持信号总强度不变，输出要乘以 (1 - ratio)。
任务（补全代码）：
下面是书中标准的层级 API 结构，请利用 NumPy（np.random.rand）补全 forward（正向传播）和 backward（反向传播）的逻辑。

In [ ]:
import numpy as np
class Dropout:
    def __init__(self, dropout_ratio=0.5):
        self.dropout_ratio = dropout_ratio
        self.mask = None # 掩膜，用来记录哪些神经元被“杀”死了

    def forward(self, x, train_flg=True):
        if train_flg:
            # 【代码填空 1】请生成一个与 x 形状相同，值为 True(存活)/False(失活) 的布尔型数组，存入 self.mask
            # 提示：使用 np.random.rand(*x.shape) 生成 0~1 的随机数，与 self.dropout_ratio 比较
            self.mask=np.random.randn(*x.shape)>self.dropout_ratio

            # 【代码填空 2】利用掩膜，将失活的神经元置为0，存活的保持原样
            return x*self.mask
        else:
            # 测试时，不丢弃，但要按比例缩放信号
            return x * (1.0 - self.dropout_ratio)

    def backward(self, dout):
        # 【代码填空 3】反向传播：只有在正向传播时存活的神经元，才能把上游梯度传下去
        # 提示：利用 self.mask 扮演一个“开关”的作用
        return dout*self.mask